#### Load data & libraries

In [6]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd
from pyspark.sql.types import IntegerType

In [7]:
pd.options.display.max_rows = None
pd.options.display.max_columns = None

In [8]:
# 1. Khởi tạo cấu hình kết nối SparkSession
spark = SparkSession.builder \
    .appName("Cleaning") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
# 2. Đọc dữ liệu trực tiếp từ HDFS
hdfs_input_path = "hdfs://localhost:9000/big_data/hotel_bookings.csv"
df = spark.read.csv(hdfs_input_path, header=True, inferSchema=True)

In [10]:
df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullab

In [11]:
df.show(5)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+----+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type| adr|required_car_parking_spaces|total_

In [12]:
df.columns

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'company',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date']

#### Type parsing

In [13]:
df = df.withColumn(
    "children", 
    F.when((F.col("children") == "NA") | F.col("children").isNull(), "0").otherwise(F.col("children"))
).withColumn(
    "children", 
    F.col("children").cast(IntegerType())
)

df.select("children").distinct().show()

+--------+
|children|
+--------+
|       1|
|       3|
|      10|
|       2|
|       0|
+--------+



#### Duplicate records (Keep all)

#### Logic check

In [14]:
# Sửa các record có previous_cancellations và previous_bookings_not_canceled là 0 nhưng is_repeated_guest lại là 1 thành 0
df_repeated_guest_corrected = df.withColumn(
    "is_repeated_guest",
    F.when(
        (F.col("is_repeated_guest") == 1) & 
        (F.col("previous_cancellations") == 0) & 
        (F.col("previous_bookings_not_canceled") == 0), 
        0
    ).otherwise(F.col("is_repeated_guest"))
)

In [15]:
df_logic = df_repeated_guest_corrected.filter(
    # 1. Bỏ phòng không có khách
    ((F.col("adults") + F.col("children") + F.col("babies")) > 0)
    &    
    # 2. Bỏ phòng 0 đêm & 0 đồng
    ~((F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights") == 0) & (F.col("adr") == 0))
    &
    # 3. Bỏ phòng có giá âm
    (F.col("adr") >= 0)
    &    
    # 4. Bỏ các trường hợp trẻ em tự đi
    ~((F.col("adults") == 0) & ((F.col("children") > 0) | (F.col("babies") > 0)))
)
print(f"Số dòng ban đầu: {df.count()}")
print(f"Số dòng sau khi làm sạch logic: {df_logic.count()}")

Số dòng ban đầu: 119390
Số dòng sau khi làm sạch logic: 118341


#### Null imputation

In [16]:
# DROP CỘT COMPANY
df_dropped = df_logic.drop("company")

# FILL NULL CHO COUNTRY VÀ AGENT
df_imputed = df_dropped.withColumn(
    "country",
    F.when(
        F.col("country").isNull() | 
        (F.col("country").cast("string") == "NULL") | 
        (F.col("country").cast("string") == "NA"), 
        "Unknown"  # Điền 'Unknown' cho quốc gia
    ).otherwise(F.col("country"))
).withColumn(
    "agent",
    F.when(
        F.col("agent").isNull() | 
        (F.col("agent").cast("string") == "NULL") | 
        (F.col("agent").cast("string") == "NA"), 
        "9999"    # Điền ID 9999 cho Đại lý
    ).otherwise(F.col("agent"))
)

print("\nKiểm tra missing values sau khi xử lý:")
df_imputed.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ["country", "agent"]
]).show()


Kiểm tra missing values sau khi xử lý:
+-------+-----+
|country|agent|
+-------+-----+
|      0|    0|
+-------+-----+



#### Outlier

In [17]:
print(f"Số dòng trước khi xóa Outlier: {df_imputed.count()}")

# Lọc bỏ tất cả những trường hợp cực đoan (Giữ lại dữ liệu bình thường)
df_cleaned = df_imputed.filter(
    (F.col("adr") < 5000) &       # Xóa giá phòng ảo tưởng (như 5400)
    (F.col("adults") <= 4) &      # Xóa khách đoàn gộp (chỉ giữ tối đa 4 người/booking)
    (F.col("children") <= 3) &    # Giữ tối đa 3 trẻ em
    (F.col("babies") <= 3)        # Giữ tối đa 3 em bé
)

print(f"Số dòng sau khi xóa Outlier: {df_cleaned.count()}")

Số dòng trước khi xóa Outlier: 118341
Số dòng sau khi xóa Outlier: 118321


#### Data export

In [18]:
df_cleaned.write.csv("hotel_bookings_cleaned.csv", header=True, mode="overwrite")